# Off_targets 

In [70]:
import random
import math
import os
import numpy as np
from itertools import combinations
import time

In [2]:
def generate_off_targets(sequence_len, mismatches):
    off_targets = []

    for mismatch_positions in combinations(range(sequence_len), mismatches):
        seq = 0
        for pos in mismatch_positions:
            seq ^= (1 << pos)  # flip to 0
        off_targets.append(seq)

    return off_targets

In [3]:
total_off_targets = generate_off_targets(20, 4)

In [4]:
for off_target in total_off_targets:
    print(f"{off_target:020b}")
print(len(total_off_targets))

00000000000000001111
00000000000000010111
00000000000000100111
00000000000001000111
00000000000010000111
00000000000100000111
00000000001000000111
00000000010000000111
00000000100000000111
00000001000000000111
00000010000000000111
00000100000000000111
00001000000000000111
00010000000000000111
00100000000000000111
01000000000000000111
10000000000000000111
00000000000000011011
00000000000000101011
00000000000001001011
00000000000010001011
00000000000100001011
00000000001000001011
00000000010000001011
00000000100000001011
00000001000000001011
00000010000000001011
00000100000000001011
00001000000000001011
00010000000000001011
00100000000000001011
01000000000000001011
10000000000000001011
00000000000000110011
00000000000001010011
00000000000010010011
00000000000100010011
00000000001000010011
00000000010000010011
00000000100000010011
00000001000000010011
00000010000000010011
00000100000000010011
00001000000000010011
00010000000000010011
00100000000000010011
01000000000000010011
1000000000000

# Metrics

### Coverage

In [5]:
true_masks = [
0b11111011010000000000,
0b00000100101111110000,
0b11000100100000001111,
0b00111000001100001110,
0b00000011010011101001,
0b10000011001100010101,
0b01111000000011010001,
0b00100110110000110010,
0b11000000011111000010,
0b01011101101000100000,
0b10010000010010111100,
0b00001111000011000110,
0b00100100111100001001,
0b01100001000100111010,
0b10101000100001100101,
0b00011010010100010011,
0b10010011101001001000,
0b01001010011000101100,
0b00110101000010000111,
0b01010001110101000100,
0b10001100000001101011,
0b11001010100010011000,
0b01010000001000110111,
0b10101001001110100000,
0b01110110110010000000,
0b00001000111001010110,
0b10010001011000101001,
0b10110111100100000000,
]

In [6]:
def check_coverage(mask_set, off_targets):
    covered = 0

    for off_target in off_targets:
        valid = False
        for mask in mask_set:
            if (off_target & mask) == 0:
                covered += 1
                valid = True
                break
        # if not valid:
            # print(f"off_target: {off_target:020b}")

    return covered / len(off_targets)


def increment_coverage(mask, coverage, remaining_off_targets):
        for off_target in remaining_off_targets:
            if (off_target & mask) == 0:
                coverage += 1
                
        return coverage / len(total_off_targets)

def remove_covered_off_targets(mask_set, remaining_off_targets):
    new_remaining = []

    for off_target in remaining_off_targets:
        covered = False
        for mask in mask_set:
            if (off_target & mask) == 0:
                covered = True
                break

        if not covered:
            new_remaining.append(off_target)

    return new_remaining

In [7]:
result = check_coverage(true_masks, total_off_targets)
result

1.0

### Overlap

In [8]:
def check_overlap(mask_set, mask, length):
    overlap_pos = []
    max_overlap = max((mask & m).bit_count() for m in mask_set)
    return max_overlap, overlap_pos

In [27]:
def get_overlap_complexity(mask_set, length):
    overlap_complexity = 0

    if len(mask_set) < 1:
        return 0
        
    for i, m1 in enumerate(mask_set):
        for j, m2 in enumerate(mask_set):
            if j <= i:
                continue
                
            for shift in range(-(length - 1), length):

                if shift >= 0:
                    shifted = m2 >> shift
                else:
                    shifted = m2 << (-shift)

                overlap = (m1 & shifted).bit_count()

                overlap_complexity += 2 ** overlap
                
    return overlap_complexity


def increment_overlap_complexity(mask_set, mask, length):
    overlap_complexity = 0
    test_set = mask_set + [mask]

    if len(test_set) < 1:
        return 0
        
    for m in mask_set:
        for shift in range(-(length - 1), length):
            if shift >= 0:
                shifted = m >> shift
            else:
                shifted = m << (-shift)

            overlap = (mask & shifted).bit_count()
            overlap_complexity += 2 ** overlap
                
    return overlap_complexity

### Position Penalties

In [28]:
mismatch_penalties = [0.0, 0.0, 0.014, 0.0, 0.0, 0.395, 0.317, 0.0, 0.389, 0.079, 0.445, 0.508, 0.613, 0.851, 0.732, 0.828, 0.615, 0.804, 0.685, 0.583]
max_position_penalty = sum(mismatch_penalties)
print(max_position_penalty)

def get_position_score(mask, mismatch_penalties):
    postition_penalty_score = 0
    for i, penalty in enumerate(mismatch_penalties):
        postition_penalty_score += (mask >> i & 1) * penalty
    return postition_penalty_score
        

7.858


# Mask Generation

In [29]:
def generate_mask(weight, length):
    pos = range(length)
    mask = 0
    # remaining_pos = set(pos) - set(overlap_pos)
    positions = random.sample(pos, weight)
    
    for pos in positions:
        mask |= (1 << pos)
    return mask

def get_neighbour(mask, length):
    for i in random.sample(range(length), 4):
        mask ^= 1 << i
    return mask

def get_neighbour_same_weight(mask, length):
    # ones = [i for i in range(length) if mask & (1 << i)]
    # zeros = [i for i in range(length) if not (mask & (1 << i))]

    # off_bit = random.choice(ones)
    # on_bit = random.choice(zeros)

    # new_mask = mask

    # # turn one off
    # new_mask ^= (1 << off_bit)

    # # turn one on
    # new_mask ^= (1 << on_bit)

    new_mask = mask
    new_mask ^= (1 << random.choice(range(length)))

    return new_mask
            

In [38]:
# def generate_mask(weight, length, mask_set):

#     counts = [1] * length

#     for mask in mask_set:
#         for i in range(length):
#             if mask & (1 << i):
#                 counts[i] += 1

#     probs = [1 / c for c in counts]

#     positions = random.choices(
#         range(length),
#         weights=probs,
#         k=weight
#     )

#     mask = 0

#     for p in set(positions):
#         mask |= (1 << p)

#     return mask

In [40]:
result = generate_mask(8, 20)
print(f"mask: {result:020b}")

result = get_neighbour(result, 20)
print(f"neighbour mask: {result:020b}")

mask: 01101010001010011000
neighbour mask: 11101010111010001000


### Set Generation

In [41]:
def generate_neighbour_set(mask_set, length):
    new_set = mask_set.copy()

    mutation_num = random.choice(range(5))

    for _ in range(mutation_num):

        option = random.random()
    
        # mutate one mask
        if option < 0.8 and len(new_set) > 0:
            idx = random.randrange(len(new_set))
            new_set[idx] = get_neighbour_same_weight(new_set[idx], length)
    
        # replace one mask
        elif option < 0.9 and len(new_set) > 0:
            idx = random.randrange(len(new_set))
            weight = new_set[idx].bit_count()
            new_set[idx] = generate_mask(weight, length)
        
        #swap mask
        elif option < 1 and len(new_set) > 0:
            idx = random.randrange(len(new_set))
            new_mask = new_set[idx] >> (20 // 2 - 1)
            #get the bottom half using 0x3FF
            new_mask |= (new_set[idx] & 0x3FF) << (20 // 2 )
    
    return new_set

# Algorithm

In [42]:
def find_lowest_covering_mask(mask_set):
    lowest_mask = 0
    lowest_coverage = 1
    for i, mask in enumerate(mask_set):
        mask_coverage = check_coverage([mask], total_off_targets)
        if mask_coverage < lowest_coverage:
            lowest_coverage = mask_coverage
            lowest_mask = mask
    return lowest_mask

In [43]:
find_lowest_covering_mask(true_masks)

1029120

In [51]:
def evaluate_mask_set(
    mask_set,
    prev_coverage,
    coverage_weight,
    overlap_weight,
    position_weight,
    specificity_weight,
):
    coverage = check_coverage(mask_set, total_off_targets)
    coverage_diff = (coverage - prev_coverage) * 100

    max_overlap = 1

    for i in range(len(mask_set)):
        for j in range(i + 1, len(mask_set)):
    
            wa = mask_set[i].bit_count()
            wb = mask_set[j].bit_count()
    
            max_overlap += max(min(wa, wb), 0.01)

    overlap = get_overlap_complexity(mask_set, 20)
    overlap_norm = overlap / (max_overlap * 20)

    position_penalty = sum(
        get_position_score(mask, mismatch_penalties)
        for mask in mask_set
    )
    position_penalty_norm = position_penalty / (max_position_penalty * len(mask_set))
    weights = [m.bit_count() for m in mask_set]
    avg_weight_norm = sum(weights) / (20 * len(weights))
    # print(avg_weight_norm)
    # min_weight_norm = min(weights) / length

    score = (
        coverage_weight * coverage
        + specificity_weight * avg_weight_norm
        - overlap_weight * overlap_norm
        - position_weight * position_penalty_norm
    )

    # print(coverage_diff)
    # print(coverage)
    # print(overlap_norm)
    # print(avg_weight_norm)
    # print(position_penalty_norm)
    # print(score)
    # print("\n")

    return score

In [55]:
def find_worst_mask(    
    mask_set,
    prev_coverage,
    coverage_weight,
    overlap_weight,
    position_weight,
    specificity_weight,
):

    worst_mask = 0
    worst_score = 99999999
    for mask in mask_set:
        score = evaluate_mask_set(
            [mask],
            prev_coverage,
            coverage_weight,
            overlap_weight,
            position_weight,
            specificity_weight
        )

        if score < worst_score:
            worst_mask = mask
            worst_score = score
            
    return worst_mask
    

In [59]:
def find_mask_set(length, num_masks=8, coverage_weight=1, overlap_weight=1, position_weight=1, specificity_weight=1, search_num=300):
    # current_set = [
    #     generate_mask(weight, length)
    #     for _ in range(num_masks)
    # ]


    best_set = true_masks.copy()
    best_score = 0

    best_coverage = 1

    T0 = 1.0
    alpha = 0.995
    count = 0

    while len(best_set) > num_masks:
        # lowest_coverage_mask = find_lowest_covering_mask(best_set)
        # best_set.remove(lowest_coverage_mask)

        worst_mask = find_worst_mask(
            best_set,
            best_coverage,
            coverage_weight,
            overlap_weight,
            position_weight,
            specificity_weight
        )
        best_set.remove(worst_mask)
            
        current_set = best_set.copy()
    
        best_coverage = check_coverage(current_set, total_off_targets)
        
        current_score = evaluate_mask_set(
            current_set,
            best_coverage,
            coverage_weight,
            overlap_weight,
            position_weight,
            specificity_weight
        )
        print(current_score)


        for iteration in range(search_num):
    
            # T = T0 / max(math.log(iteration + 1), 0.000001)
            T = max(T0 * (alpha ** iteration), 0.001, )
    
            candidate_set = generate_neighbour_set(
                current_set,
                length
            )
    
            candidate_score = evaluate_mask_set(
                candidate_set,
                best_coverage,
                coverage_weight,
                overlap_weight,
                position_weight,
                specificity_weight
            )
    
            diff = candidate_score - current_score
    
            accept = False
    
            if diff > 0:
                accept = True
            else:
                probability = math.exp(diff / T)
    
                if random.random() < probability:
                    accept = True
    
            if accept:
                current_set = candidate_set
                best_coverage = check_coverage(current_set, total_off_targets)
                current_score = candidate_score
    
            if current_score > best_score:
                best_set = current_set.copy()
                best_score = current_score
                count = 0
                print(best_score)
    
                best_coverage = check_coverage(best_set, total_off_targets)
                
            else:
                count += 1

    return best_set
    

In [66]:
start_time = time.perf_counter()
mask_set = find_mask_set(
    length = 20, 
    num_masks = 4, 
    coverage_weight = 2,
    overlap_weight = 1, 
    position_weight = 0.2,
    specificity_weight = 8,
    search_num = 3000
)
end_time = time.perf_counter() 

elapsed_time = end_time - start_time
print(elapsed_time)

3.91559888896237
3.886919863395928
3.8882817334479407
3.892794147710655
3.893611481377022
3.893828945849082
3.8951695396589012
3.8975077489529477
3.8994331554572765
3.899987098218937
3.9025741125024576
3.906724335007602
3.911549733669894
3.9140866411990016
3.9191894221487593
3.9269139850477046
3.928928731630281
3.9290757683007413
3.934482304684181
3.9378075174296936
3.9383059252022576
3.947111939259354
3.938361239047306
3.924866998463633
3.9047534190566444
3.889424649380026
3.847951139743284
3.8284309957130622
3.799262331725976
3.7597317147094946
3.709536564615353
3.669049318859885
3.6354015153097543
3.590480698338804
3.5297584875818715
3.4790782496286945
3.42108281589792
3.3766857333783946
3.311887626410556
3.2161582768661643
3.092419091085551
3.017482865543049
2.8985105126611517
2.791059638573597
2.6684420180042383
158.91026457299995


In [67]:
for m in mask_set:
    print(f"mask: {m:020b}")
result = check_coverage(mask_set, total_off_targets)
print(result)

mask: 00010010010010010111
mask: 10100000000010110111
mask: 00000100101101010110
mask: 11000101100000101011
0.3234262125902993


In [71]:
os.makedirs("makssubsets", exist_ok=True)

with open("masksubsets/testmask2-4-20.txt", "w") as f:
    for m in mask_set:
        f.write(f"{m:020b}\n")

In [66]:
test_masks = [
0b10001000000000101000,
0b01100010101001010001,
0b10100100100100000111,
0b00000000000000000000
]

In [67]:
check_coverage(test_masks, total_off_targets)

1.0

#### 